##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Social Media Intelligence with XPOZ MCP

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Social_media_intelligence_with_XPOZ_MCP.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook shows how to use [XPOZ](https://xpoz.ai) — a social-intelligence MCP server with 1.5B+ indexed posts across Twitter, Instagram, Reddit & TikTok — to build a real-time social media analysis pipeline with the Gemini API.

You will learn to:
- Connect to a remote MCP server via HTTP (JSON-RPC)
- Fetch cross-platform social data (Twitter, Reddit, Instagram)
- Use Gemini to produce structured sentiment analysis from raw social posts

## Setup

### Install the SDK

In [ ]:
!pip install -q google-genai httpx

### Set up your API keys

You need two API keys stored as [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb):
- `GOOGLE_API_KEY` — from [Google AI Studio](https://aistudio.google.com/)
- `XPOZ_API_KEY` — free at [xpoz.ai](https://xpoz.ai) (100K results/month)

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
XPOZ_API_KEY = userdata.get("XPOZ_API_KEY")

### Choose a model

In [ ]:
MODEL_ID = "gemini-3-flash-preview"  # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}

In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

## Connect to XPOZ MCP server

XPOZ provides social data via the [Model Context Protocol](https://modelcontextprotocol.io/). The server accepts JSON-RPC requests over HTTP. Create a lightweight client that initializes a session and calls tools like `countTweets` and `getTwitterPostsByKeywords`.

In [ ]:
import json
import re
import csv
import io
import httpx
from datetime import datetime, timedelta

XPOZ_URL = "https://mcp.xpoz.ai/mcp"
_msg_id = 0
_session_id = None

In [ ]:
def _parse_sse(text):
    """Extract JSON-RPC result from an SSE response."""
    result = {}
    for line in text.split("\n"):
        if line.startswith("data: "):
            try:
                result = json.loads(line[6:])
            except json.JSONDecodeError:
                pass
    return result


async def _mcp_post(http_client, payload):
    """POST a JSON-RPC request and handle JSON or SSE responses."""
    global _session_id
    headers = {
        "Authorization": f"Bearer {XPOZ_API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if _session_id:
        headers["Mcp-Session-Id"] = _session_id
    resp = await http_client.post(XPOZ_URL, json=payload, headers=headers)
    if "mcp-session-id" in resp.headers:
        _session_id = resp.headers["mcp-session-id"]
    ct = resp.headers.get("content-type", "")
    if "text/event-stream" in ct:
        return _parse_sse(resp.text)
    elif resp.text.strip():
        return resp.json()
    return {}

In [ ]:
async def _ensure_session(http_client):
    """Initialize the MCP session if not already active."""
    global _session_id
    if _session_id:
        return
    await _mcp_post(http_client, {
        "jsonrpc": "2.0", "id": 0, "method": "initialize",
        "params": {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "cookbook", "version": "1.0"},
        },
    })
    await _mcp_post(http_client, {
        "jsonrpc": "2.0", "method": "notifications/initialized",
    })

In [ ]:
def _parse_xpoz(text):
    """Parse XPOZ compact tabular format into a Python dict."""
    result = {"success": "success: true" in text, "data": {}}
    hdr = re.search(r"results\[\d+\]\{([^}]+)\}:", text)
    if hdr:
        fields = hdr.group(1).split(",")
        rows = []
        for line in text[hdr.end():].split("\n"):
            if not line.startswith("    "):
                if rows or (line.strip() and not line.startswith(" ")):
                    break
                continue
            try:
                vals = next(csv.reader(io.StringIO(line.strip())))
                row = {}
                for i, f in enumerate(fields):
                    if i < len(vals):
                        v = vals[i].strip()
                        if v in ("null", ""):
                            row[f] = None
                        else:
                            try:
                                row[f] = int(v)
                            except ValueError:
                                try:
                                    row[f] = float(v)
                                except ValueError:
                                    row[f] = v
                rows.append(row)
            except Exception:
                pass
        result["data"]["results"] = rows
    for m in re.finditer(r"^\s{2}(\w+):\s+(.+)$", text, re.MULTILINE):
        k, v = m.group(1), m.group(2).strip()
        if k.startswith("results"):
            continue
        try:
            result["data"][k] = int(v)
        except ValueError:
            try:
                result["data"][k] = float(v)
            except ValueError:
                result["data"][k] = v
    return result

In [ ]:
async def call_xpoz(tool_name, params):
    """Call an XPOZ MCP tool and return parsed results."""
    global _msg_id
    _msg_id += 1
    async with httpx.AsyncClient(timeout=120) as http_client:
        await _ensure_session(http_client)
        data = await _mcp_post(http_client, {
            "jsonrpc": "2.0", "id": _msg_id,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": params},
        })
    if "error" in data:
        print(f"MCP error: {data['error']}")
        return {"data": {"results": []}}
    if "result" not in data:
        print(f"Unexpected response: {str(data)[:200]}")
        return {"data": {"results": []}}
    content = data.get("result", {}).get("content", [{}])
    text = content[0].get("text", "{}") if content else "{}"
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return _parse_xpoz(text)

Verify the connection works.

In [ ]:
def days_ago(n):
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")

test = await call_xpoz("countTweets", {"phrase": "Bitcoin", "startDate": days_ago(1)})
print(f"Connected to XPOZ MCP \u2014 Bitcoin tweets in last 24h: {test['data'].get('count', 'ok'):,}")

## Fetch social data

Analyze **Bitcoin** social sentiment across three platforms. XPOZ provides separate tools for each:

| Tool | Platform | Returns |
|---|---|---|
| `countTweets` | Twitter | Total volume |
| `getTwitterPostsByKeywords` | Twitter | Post text, author, engagement |
| `getRedditPostsByKeywords` | Reddit | Title, selftext, score |
| `getInstagramPostsByKeywords` | Instagram | Caption, username, likes |

In [ ]:
count_r = await call_xpoz("countTweets", {
    "phrase": "Bitcoin", "startDate": days_ago(7),
})
tweet_count = count_r["data"].get("count", 0)
print(f"7-day Bitcoin tweet volume: {tweet_count:,}")

In [ ]:
# Two batches to get ~600 tweets (API caps at 300 per call)
tw1 = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "Bitcoin OR BTC", "startDate": days_ago(3), "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"],
})
tw2 = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "Bitcoin OR BTC", "startDate": days_ago(7), "endDate": days_ago(3), "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"],
})
tweets = tw1["data"].get("results", []) + tw2["data"].get("results", [])
print(f"Twitter: {len(tweets)} posts")

In [ ]:
rd_r = await call_xpoz("getRedditPostsByKeywords", {
    "query": "Bitcoin OR BTC", "startDate": days_ago(7), "limit": 300,
    "fields": ["title", "selftext", "authorUsername", "score", "subredditName", "createdAtDate"],
})
reddit_posts = rd_r["data"].get("results", [])
print(f"Reddit: {len(reddit_posts)} posts")

In [ ]:
ig_r = await call_xpoz("getInstagramPostsByKeywords", {
    "query": "Bitcoin", "limit": 100,
    "fields": ["caption", "username", "likeCount", "commentCount", "createdAtDate"],
})
ig_posts = ig_r["data"].get("results", [])
print(f"Instagram: {len(ig_posts)} posts")

In [ ]:
total_posts = len(tweets) + len(reddit_posts) + len(ig_posts)
print(f"Total: {total_posts} posts sampled from {tweet_count:,} indexed tweets")

## Analyse with Gemini

Feed the social data to Gemini and request a structured sentiment analysis.

In [ ]:
def _safe_int(v):
    try:
        return int(v) if v else 0
    except (ValueError, TypeError):
        return 0

tw_text = "\n".join(
    f"@{p.get('authorUsername', '?')}: {p.get('text', '')} "
    f"[likes:{_safe_int(p.get('likeCount'))}, RTs:{_safe_int(p.get('retweetCount'))}]"
    for p in tweets[:200]
)
rd_text = "\n".join(
    f"r/{p.get('subredditName', '')} \u2014 {p.get('title', '')}: "
    f"{(p.get('selftext') or '')[:200]} [score:{_safe_int(p.get('score'))}]"
    for p in reddit_posts[:100]
)
ig_text = "\n".join(
    f"@{p.get('username', '?')}: {(p.get('caption') or '')[:200]} "
    f"[likes:{_safe_int(p.get('likeCount'))}]"
    for p in ig_posts[:50]
)

In [ ]:
prompt = f"""Analyse the social-media sentiment for Bitcoin based on these posts from the last 7 days.
Total tweet volume: {tweet_count:,}

=== Twitter ({len(tweets)} posts) ===
{tw_text}

=== Reddit ({len(reddit_posts)} posts) ===
{rd_text}

=== Instagram ({len(ig_posts)} posts) ===
{ig_text}

Return your analysis in EXACTLY this format:

OVERALL SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

KEY THEMES:
1. <theme> \u2014 <sentiment> \u2014 <explanation>
2. \u2026
3. \u2026
4. \u2026
5. \u2026

RISK SIGNALS:
- <risk>
- \u2026

OPPORTUNITIES:
- <opportunity>
- \u2026

TOP POSTS:
1. <platform> @<author>: <quote> \u2014 <why it matters>
2. \u2026
3. \u2026"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)
analysis = response.text
print(analysis)

## Results

Look at the highest-engagement posts that fed into the analysis.

In [ ]:
print("Top tweets by engagement:\n")
for t in sorted(tweets, key=lambda x: _safe_int(x.get("likeCount")), reverse=True)[:10]:
    likes = _safe_int(t.get("likeCount"))
    rts = _safe_int(t.get("retweetCount"))
    text = (t.get("text") or "")[:120]
    print(f"  @{t.get('authorUsername', '?')} ({likes:,} likes, {rts:,} RTs)")
    print(f"    {text}")
    print()

In [ ]:
print("Top Reddit posts by score:\n")
for p in sorted(reddit_posts, key=lambda x: _safe_int(x.get("score")), reverse=True)[:5]:
    score = _safe_int(p.get("score"))
    print(f"  r/{p.get('subredditName', '?')} \u2014 {p.get('title', '')} (score: {score:,})")
    print()

## Next steps

This notebook demonstrated fetching cross-platform social data via XPOZ MCP and analyzing it with Gemini. You can extend this pattern to:

- Add TikTok data via `getTiktokPostsByKeywords`
- Track sentiment over time by running daily
- Compare brands with parallel queries (e.g., Bitcoin vs Ethereum)
- Use [function calling](https://ai.google.dev/gemini-api/docs/function-calling) to let Gemini call XPOZ tools directly

See the [XPOZ documentation](https://xpoz.ai) for the full list of available MCP tools.